### 讀取字典、算分數、擷取片段
先遍歷所有新聞文本，把需要的片段擷取下來(只含供應鏈風險詞彙相關的)=>存在llm_ready_data

1. 只擷取有供應鏈風險分數的文章
2. 用供應鏈那個句子下來做分類


In [ ]:
# 設定路徑
year = 2012
sc_path = r'C:\Users\user\Desktop\ravenpack\RP_SCrisk\1_dictionary\供應鏈字典\full_bigrams_cleaned.csv'
risk_path = r'C:\Users\user\Desktop\ravenpack\RP_SCrisk\1_dictionary\風險字典_hassan\risk.csv'
txt_base_path = rf'C:\Users\user\Desktop\ravenpack\DATA\target_data\{year}'
input_master_csv = rf"C:\Users\user\Desktop\ravenpack\DATA\target_data\篩選Risk keywords的data\All_Years_Grand_Total_Riskwords_{year}.csv"  # 總檔CSV


# 主題模型輸出檔名修正
output_csv = rf'C:\Users\user\Desktop\ravenpack\RP_SCrisk\2_calculate_supply_chain_risk_and_extract_segments\output\llm_ready_整年\{year}\llm_ready_data_{year}_context50_snippetlevel.csv'

# 新增公司層級的輸出檔名
article_company_output_csv = rf'C:\Users\user\Desktop\ravenpack\RP_SCrisk\2_calculate_supply_chain_risk_and_extract_segments\output\llm_ready_整年\{year}\article_level_scores_{year}.csv'
import os
os.makedirs(os.path.dirname(output_csv), exist_ok=True)
os.makedirs(os.path.dirname(article_company_output_csv), exist_ok=True)     

In [ ]:
import os
import re
import ast
import pandas as pd
import numpy as np



def load_dictionaries(sc_csv_path, risk_csv_path):
    """載入供應鏈權重字典與風險詞集合"""
    # 處理 SC Dictionary
    df_sc = pd.read_csv(sc_csv_path)
    sc_weights = {}
    for _, row in df_sc.iterrows():
        raw_bigram = row['bigram']
        weight = row['weight']
        try:
            # 支援 tuple 字串 "('supply', 'chain')" 或 "supply_chain"
            if '(' in str(raw_bigram):
                key = ast.literal_eval(raw_bigram)
            elif '_' in str(raw_bigram):
                parts = raw_bigram.lower().split('_')
                key = (parts[0], parts[1])
            else: continue
            sc_weights[key] = weight
        except: continue

    # 處理 Risk Dictionary
    df_risk = pd.read_csv(risk_csv_path)
    risk_set = set(df_risk['Word'].str.lower().str.strip())
    print(list(risk_set)[:10])
    return sc_weights, risk_set


def process_single_txt(file_path):
    """讀取並清洗文本，回傳標記化的 bigrams"""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            raw_text = f.read()
        text_clean = re.sub(r'[^a-zA-Z\s]', ' ', raw_text).lower()
        tokens = text_clean.split()
        bigram_sequence = list(zip(tokens, tokens[1:]))

        #將切分好的bigrams加上index，回傳 [(0, (w1, w2)), (1, (w2, w3)), ...] 的格式，以及 bigram 的總數量
        return list(enumerate(bigram_sequence)), len(bigram_sequence)
    except Exception as e:
        print(f"讀取檔案 {file_path} 失敗: {e}")
        return [], 0


def calculate_risk_and_extract_context(indexed_bigrams,
                                       sc_weights,
                                       risk_set,
                                       total_count,
                                       score_window=10,
                                       context_window=50):
    """核心邏輯：計算分數 + 擷取 LLM 專用上下文"""
    # 1. 找出風險詞出現的位置(檢查 bigram 中的任一詞是否在 risk_set 中)，是的話就記錄該 bigram 的 index
    risk_indices = [
        idx for idx, (w1, w2) in indexed_bigrams
        if w1 in risk_set or w2 in risk_set
    ]

    # 如果沒有風險詞，直接回傳 0 分和空上下文
    if not risk_indices:
        return 0.0, 0.0, [], [], [] # 沒有風險詞，分數為 0，LLM 上下文為空

    # 2. 計算分數 (使用 window=10 邏輯)
    valid_score_indices = set()
    for r in risk_indices:
        for i in range(max(0, r - score_window), min(total_count - 1, r + score_window) + 1):
            valid_score_indices.add(i)

    # 若 bigram 的 index 在 valid_score_indices 中，且該 bigram 存在於 sc_weights 中，就將其權重加入加權總和
    weighted_sum = 0.0
    for index, bigram_tuple in indexed_bigrams:
        if index in valid_score_indices and bigram_tuple in sc_weights:
            weighted_sum += sc_weights[bigram_tuple]


    # 最終分數 = 加權總和 / bigram 總數量 (如果 bigram 總數量 > 0，否則為 0)
    final_score = weighted_sum / total_count if total_count > 0 else 0.0

    # 3. 擷取 LLM 內容：針對每一個符合的供應鏈詞彙獨立建立[start, end]範圍
    # 先找出所有符合條件的 bigram 的 index
    matched_sc_indices = [
        index for index, bigram_tuple in indexed_bigrams
        if index in valid_score_indices and bigram_tuple in sc_weights
    ]

    # 如果沒有符合條件的供應鏈詞彙，回傳分數與空列表
    if not matched_sc_indices:
        return final_score, weighted_sum, [], [], []

    # 4. 收集所有供應鏈詞彙的獨立前後 window 區間
    snippets = []
    matched_bigrams = [] #新增變數用於收集命中的bigrams

    for idx in matched_sc_indices:
        start = max(0, idx - context_window)
        end = min(total_count - 1, idx + context_window)

        # 將區間的文字加入 snippets
        seg = [indexed_bigrams[i][1][0] for i in range(start, end + 1)]
        seg.append(indexed_bigrams[end][1][1])
        snippets.append(" ".join(seg))

        # 新增：紀錄該 snippet 對應的命中 bigrams
        matched_bigrams.append(indexed_bigrams[idx][1])


    return final_score, weighted_sum, snippets, matched_sc_indices, matched_bigrams




In [ ]:
# 執行字典載入與處理
sc_weights, risk_set = load_dictionaries(sc_path, risk_path)

In [ ]:
import numpy as np
from collections import defaultdict

# 讀取總檔 CSV
print("\n" + "=" * 60)
print("步驟 2: 讀取總檔 CSV")
print("=" * 60)
df_master_raw = pd.read_csv(input_master_csv)
before_dedup = len(df_master_raw)

# ---- 公司層級版本:一篇文章可以對應多家公司 ----
df_master_entity = df_master_raw.drop_duplicates(subset=['RP_DOCUMENT_ID', 'RP_ENTITY_ID']).reset_index(drop=True)


# ---- 文章/snippet 層級版本:一篇文章只留一列(給文字處理用，跟公司無關) ----
df_master = df_master_raw.drop_duplicates(subset=['RP_DOCUMENT_ID']).reset_index(drop=True)
print(f"去重：{before_dedup} → 文章層級 {len(df_master)} 筆，公司層級 {len(df_master_entity)} 筆")

# 檢查必要欄位
if 'RP_DOCUMENT_ID' not in df_master.columns:
    raise ValueError(" CSV 總檔必須包含 'RP_DOCUMENT_ID' 欄位！")

print(f"前5筆檔案: {df_master['RP_DOCUMENT_ID'].head().tolist()}")


# 將 EVENT_SIMILARITY_DAYS 與 TITLE_SIMILARITY_DAYS 轉為數值型態，若無法轉換則設為 NaN
df_master_entity['EVENT_SIMILARITY_DAYS'] = pd.to_numeric(df_master_entity['EVENT_SIMILARITY_DAYS'], errors='coerce')
df_master_entity['TITLE_SIMILARITY_DAYS'] = pd.to_numeric(df_master_entity['TITLE_SIMILARITY_DAYS'], errors='coerce')

# 新增：判斷是否第一次被提及
df_master_entity['event_detected'] = df_master_entity['TYPE'].notna() & (df_master_entity['TYPE'].astype(str).str.strip() != '')
df_master_entity['is_first_event_mention'] = df_master_entity['EVENT_SIMILARITY_DAYS'] >= 364.99
df_master_entity['is_first_title_mention'] = df_master_entity['TITLE_SIMILARITY_DAYS'] >= 364.99

# event 或 title 任一為首次提及即可
df_master_entity['is_first_mention'] = (
    df_master_entity['is_first_event_mention'] | df_master_entity['is_first_title_mention']
)

# mention_basis 記錄「是哪一邊(或兩邊)判定為首次提及」
df_master_entity['mention_basis'] = np.select(
    [
        df_master_entity['is_first_event_mention'] & df_master_entity['is_first_title_mention'],
        df_master_entity['is_first_event_mention'],
        df_master_entity['is_first_title_mention'],
    ],
    ['both', 'event', 'title'],
    default='none'
)

# 算該 chain 下有幾個不同 RP_DOCUMENT_ID
chain_size = df_master_entity.groupby('PROVIDER_DOCUMENT_CHAIN_ID')['RP_DOCUMENT_ID'].transform('nunique')
df_master_entity['chain_update_count'] = chain_size
df_master_entity['is_chain_update'] = chain_size > 1

# 計算使用的相似性天數
# 搭配 OR 邏輯改取兩者較大值，novelty_bucket 才會跟 is_first_mention 一致
df_master_entity['similarity_days_used'] = df_master_entity[
    ['EVENT_SIMILARITY_DAYS', 'TITLE_SIMILARITY_DAYS']
].max(axis=1)
bins = [-0.01, 1, 7, 30, 90, 364.99, 365.01]
labels = ['same_day', 'within_week', 'within_month', 'within_quarter', 'within_year', 'first_ever']
df_master_entity['novelty_bucket'] = pd.cut(df_master_entity['similarity_days_used'], bins=bins, labels=labels)


# 把這些欄位一起放進 doc_to_entities
doc_cols = ['RP_ENTITY_ID', 'ENTITY_NAME', 'event_detected', 'is_first_mention',
            'mention_basis', 'is_chain_update', 'chain_update_count', 'novelty_bucket']

doc_to_entities = defaultdict(list)
for doc_id, *vals in df_master_entity[['RP_DOCUMENT_ID'] + doc_cols].itertuples(index=False, name=None):
    doc_to_entities[doc_id].append(dict(zip(doc_cols, vals)))
doc_to_entities = dict(doc_to_entities)

sample_id = list(doc_to_entities.keys())[0]
print("sample:", sample_id, "->", doc_to_entities[sample_id])

In [ ]:
# 列出欄位
df_master.columns.tolist()
# 印出特定欄位的前5筆資料
print(df_master[['EVENT_SIMILARITY_DAYS']].head())

In [ ]:
import pickle

cache_dir = os.path.dirname(output_csv)
os.makedirs(cache_dir, exist_ok=True)  # 確保資料夾一定存在，不會因為路徑缺失而寫入失敗
cache_path = os.path.join(cache_dir, 'file_index_cache.pkl')

if os.path.exists(cache_path):
    with open(cache_path, 'rb') as f:
        file_cache = pickle.load(f)
    print(f"從快取讀取索引，共 {len(file_cache)} 個檔案")
else:
    file_cache = {}
    for root, dirs, files in os.walk(txt_base_path):
        for file in files:
            if file.endswith('.txt'):
                file_cache[file.replace('.txt', '')] = os.path.join(root, file)
    with open(cache_path, 'wb') as f:
        pickle.dump(file_cache, f)
    print(f"已建立並儲存索引，共 {len(file_cache)} 個檔案")

針對csv檔案去做llm_ready

In [ ]:
import pickle


# 處理文章
print("\n" + "=" * 60)
print("步驟 3: 處理文章並擷取片段")
print("=" * 60)

# 用於主題模型的檔案
results = []

# 用於公司層級的分數檔案
article_results = []
article_company_total_saved = 0


batch_size = 200

# 用於 output_csv 實際累積的 snippet 列數
total_snippet_rows = 0

not_found_files = []
processed_count = 0
zero_score_count = 0  # 計算有內容但無符合snippet的片段(即風險分數為0)
empty_bigram_count = 0 # 計算檔案全部為空的篇章

# 加入主題模型用的檔案初始化邏輯
processed_files = set()
if os.path.exists(output_csv):
    try:
        # 讀取現有的 CSV，只拿 ID 欄位以節省記憶體
        existing_df = pd.read_csv(output_csv, usecols=['file_name'])
        processed_files = set(existing_df['file_name'].astype(str).tolist())
        total_snippet_rows = len(existing_df)
        print(f"偵測到已存在的檔案，目前已有 {total_snippet_rows} 列 snippet")
    except Exception as e:
        print(f"讀取舊檔案失敗或檔案為空: {e}")
        processed_files = set()
        total_snippet_rows = 0
        
        
# 公司層級分數檔案初始化邏輯
# 除了紀錄已經處理過的檔名，也連同分數一起讀出來，0分文張才能被判定為已經完成，不用重新讀檔案計分
processed_article_files = set()
processed_article_scores = {}
if os.path.exists(article_company_output_csv):
    try:
        existing_article_df = pd.read_csv(article_company_output_csv, usecols=['file_name', 'sc_risk_score'])
        existing_article_df['file_name'] = existing_article_df['file_name'].astype(str)
        processed_article_scores = dict(zip(existing_article_df['file_name'], existing_article_df['sc_risk_score']))
        processed_article_files = set(existing_article_df['file_name'].astype(str).tolist())
        article_company_total_saved = len(processed_article_files)
        print(f"偵測到已存在的公司層級檔案，將跳過 {article_company_total_saved} 篇已記錄的文章。")
    except Exception as e:
        print(f"讀取舊的公司層級檔案失敗或檔案為空: {e}")
        processed_article_files = set()
        processed_article_scores = {}
        article_company_total_saved = 0


print(f"沿用已存在的索引，共 {len(file_cache)} 個檔案")


skipped_count = 0
articles_missing_entities = []  # 觀察用：doc_to_entities 找不到公司資料的文章


for idx, row in df_master.iterrows():
    filename = row['RP_DOCUMENT_ID']  # 檔名欄位

    #  檢查是否已處理過
    #  - 公司層級檔案有這篇文章的分數紀錄 且 
    #  - 主題檔案已經有這篇文章，或者公司層級檔案紀錄的分數為0
    # 兩者同時滿足才整篇跳過    
    if filename in  processed_article_files and (
        filename in processed_files or processed_article_scores[filename] == 0
    ):
        skipped_count += 1
        continue

    # 從快取中查找檔案
    if filename not in file_cache:
        not_found_files.append(filename)
        continue

    file_path = file_cache[filename]

    event_type = row.get('TYPE', '') # 從總表取得 TYPE 欄位
    # 若TYPE欄位為空則標記為unknown
    if pd.isna(event_type):
        event_type = 'unknown'
    

    # 處理文本
    indexed_bg, total_cnt = process_single_txt(file_path)
    
    
    # 1. 先算此篇文章的分數
    if total_cnt > 0:
        score, w_sum, snippets, matched_indices, matched_bigrams = calculate_risk_and_extract_context(
            indexed_bg, sc_weights, risk_set, total_cnt)
    
    else:
        empty_bigram_count +=1
        score, w_sum, snippets, matched_indices, matched_bigrams = 0.0, 0.0, [], [], []
        
    
    # 2. 公司層級的分數檔案
    entities_for_doc = doc_to_entities.get(filename, [])
    if not entities_for_doc:
        articles_missing_entities.append(filename)

    for entity in entities_for_doc:
        article_results.append({
            'TIMESTAMP_UTC': row.get('TIMESTAMP_UTC', ''),
            'event_type': event_type,
            'file_name': row['RP_DOCUMENT_ID'],
            'RP_ENTITY_ID': entity['RP_ENTITY_ID'], # 新增：公司ID
            'ENTITY_NAME': entity['ENTITY_NAME'], # 新增：公司名稱
            'sc_risk_score': score,
            'weighted_sum': w_sum,
            'total_bigrams': total_cnt,
            # 新增欄位
            'is_first_mention': entity.get('is_first_mention', pd.NA),
            'mention_basis': entity.get('mention_basis', ''),
            'is_chain_update': entity.get('is_chain_update', False),
            'chain_update_count': entity.get('chain_update_count', 1),
            'novelty_bucket': entity.get('novelty_bucket', ''),
        })

    processed_article_files.add(filename)
    processed_article_scores[filename] = score

    # 3. 批次寫入公司層級分數檔案
    if len(article_results) >= batch_size:
        df_article_batch = pd.DataFrame(article_results)  
        mode = 'w' if article_company_total_saved == 0 else 'a'    
        header = (article_company_total_saved == 0)
        df_article_batch.to_csv(article_company_output_csv,        
                                mode=mode,
                                header=header,
                                index=False,
                                encoding='utf-8-sig')

        article_company_total_saved += len(article_results)
        print(f"\n 已追加 {len(article_results)} 筆記錄至 {article_company_output_csv}（累計已儲存 {article_company_total_saved} 筆文章列）\n")
        article_results = [] 
    

    # 4. 主題模型：只有 score > 0、有 snippets、且這篇文章尚未寫進主題檔才寫進 results
    if score > 0 and snippets and filename not in processed_files:
        #遍歷這篇文章拆出來的每一個獨立 snippet，各自建立一列資料
        for s_idx, snippet in enumerate(snippets):
            results.append({
                'TIMESTAMP_UTC': row.get('TIMESTAMP_UTC', ''),
                'event_type': event_type,
                'file_name': row['RP_DOCUMENT_ID'],  # 使用原始檔名（不含路徑）
                'sc_risk_score': score,
                'weighted_sum': w_sum,
                'total_bigrams': total_cnt,
                'snippet_index': s_idx,    # 片段在該文章中的序號
                'matched_index': matched_indices[s_idx],  # 該 snippet 命中的 bigram 在全文的index
                'matched_bigram': "_".join(matched_bigrams[s_idx]),  #　修改：改成跟字典檔一致的底線字串格式(word1_word2)
                'llm_input_text': snippet,
            })
        processed_files.add(filename)
        processed_count += 1
        print(
            f"✅ [{processed_count}] 已擷取: {filename} (事件類型: {event_type}, 分數: {score:.4f}, 拆分 Snippets 數: {len(snippets)})"
        )
        

        # 每達到 batch_size 就追加到同一個文件
        if len(results) >= batch_size:
            df_batch = pd.DataFrame(results)

            # 第一次寫入包含表頭，之後追加不含表頭
            mode = 'w' if total_snippet_rows == 0 else 'a'
            header = (total_snippet_rows == 0)
            df_batch.to_csv(output_csv,
                            mode=mode,
                            header=header,
                            index=False,
                            encoding='utf-8-sig')

            total_snippet_rows += len(results)
            print(f"\n 已追加 {len(results)} 筆記錄至 {output_csv}（累計已儲存 {total_snippet_rows} 筆 Snippet 列）\n")
            results = []
            
    
    elif total_cnt > 0 and (score == 0 or not snippets):
        zero_score_count += 1
        print(
            f"⚠️ [{zero_score_count}] {filename} 的供應鏈風險分數為 0，或無符合條件的 snippet，已跳過。"
        )


# 保存最後剩餘的數據(主題模型)
if results:
    df_batch = pd.DataFrame(results)
    mode = 'a' if total_snippet_rows > 0 else 'w'
    header = (total_snippet_rows == 0)
    df_batch.to_csv(output_csv, mode=mode, header=header,index=False, encoding='utf-8-sig')
    total_snippet_rows += len(results)
    print(f"\n 已追加最後 {len(results)} 筆（總計 {total_snippet_rows} Snippets 列）")
    
    
# 保存數據(公司層級)
if article_results:
    df_article_batch = pd.DataFrame(article_results)
    mode = 'a' if article_company_total_saved > 0 else 'w'
    header = (article_company_total_saved == 0)
    df_article_batch.to_csv(article_company_output_csv, mode=mode, header=header,index=False, encoding='utf-8-sig')
    article_company_total_saved += len(article_results)
    print(f"\n 已追加最後 {len(article_results)} 筆（總計 {article_company_total_saved}篇文章）")
    

# 輸出統計報告
print("\n" + "=" * 60)
print("處理完畢！主題模型檔案統計報告")
print("=" * 60)
print(f"總檔記錄數: {len(df_master)}")
print(f"跳過已處理: {skipped_count} 篇")
print(f"成功處理: {processed_count} 文章數量")
print(f"找不到檔案: {len(not_found_files)} 篇")
print(f"最終輸出: {total_snippet_rows} 列 snippets ")
print(f"主題結果已儲存至: {output_csv}")
print("=" * 60)
print("公司層級文章統計報告")
print(f"公司層級結果已儲存至: {article_company_output_csv}")
print(f"供應鏈風險分數為 0 的文章: {zero_score_count} 篇")
print(f"清洗後無有效 bigrams 的文章: {empty_bigram_count} 篇")



if not_found_files:
    print(f"\n 以下 {len(not_found_files)} 個檔案找不到:")
    for missing_file in not_found_files[:10]:  # 只顯示前10個
        print(f"   - {missing_file}")
    if len(not_found_files) > 10:
        print(f"   ... 還有 {len(not_found_files) - 10} 個檔案")